# 남한 시군구 경계 지도 생성기

이 노트북은 남한 지도를 시·군 단위로 그려 1m × 1.391m 크기의 고해상도(1000dpi) 이미지로 출력합니다.

## 기능
- 시·군 경계선 + 시·군 이름 표기
- 광역시(서울·부산·대구·인천·광주·대전·울산)는 구 경계 유지
- 도는 시/군 단위로 병합 (구로 나뉜 시는 시 이름으로 병합)
- 시도 경계는 두껍게, 도 이름은 흐리게 2배 크기로 표기
- 북한 하단 형태 포함 (절단선 아래만)

## 필요한 파일
- `skorea-municipalities-2018-geo.json` : 남한 시군구 경계 데이터
- `north_korea.geojson` : 북한 형태 데이터

## 필요한 패키지
```
pip install matplotlib shapely numpy
```

In [ ]:
# 1. 패키지 임포트 및 설정
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from shapely.geometry import shape, box
from shapely.ops import unary_union

# 한글 폰트
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

print("설정 완료")

In [ ]:
# 2. 파일 경로 및 설정값
DATA = r"skorea-municipalities-2018-geo.json"
NK = r"north_korea.geojson"
OUT = r"south_korea_map.png"

# 광역시/특별시 코드 (구 경계 유지)
METRO_CODES = {"11", "21", "22", "23", "24", "25", "26"}

# 시도 전체 이름
SIDO_NAMES = {
    "11": "서울특별시", "21": "부산광역시", "22": "대구광역시", "23": "인천광역시",
    "24": "광주광역시", "25": "대전광역시", "26": "울산광역시", "29": "세종특별자치시",
    "31": "경기도", "32": "강원도", "33": "충청북도", "34": "충청남도",
    "35": "전라북도", "36": "전라남도", "37": "경상북도", "38": "경상남도",
    "39": "제주특별자치도",
}

# 출력 크기 (인치) : 가로 1m, 세로는 실제 지리 비율(1.391)에 맞춤 (북한 일부)
W_INCH = 39.37          # 1m
H_INCH = W_INCH * 1.391 # 1.391m (실제 지리 비율, 왜곡 없음)
DPI = 1000  # 인화용 초고해상도

# 북한 절단 위도 (이 위도 아래만 표시)
NK_CUT = 38.71740371170524

print("설정값 로드 완료")

In [ ]:
# 3. 데이터 로드 및 폴리곤 준비
with open(DATA, encoding="utf-8") as f:
    gj = json.load(f)

features = gj["features"]

# 남한 시군구 (울릉도/독도 제외)
polys = []          # 채우기용 폴리곤 (광역시 구 + 도 시/군)
names = {}          # 이름 표기 대상 (시/군 이름)
sido_groups = {}    # 시도 코드 -> 폴리곤 리스트 (시도 경계용)
city_groups = {}    # 도: 시/군 이름 -> 폴리곤 리스트 (병합용)
for ft in features:
    name = ft["properties"]["name"]
    code = ft["properties"]["code"]
    if name == "울릉군":
        continue  # 울릉도/독도 제외
    geom = shape(ft["geometry"])
    sido_groups.setdefault(code[:2], []).append(geom)
    if code[:2] in METRO_CODES:
        # 광역시/특별시: 구 경계 유지 (이름은 아래에서 병합 후 표기)
        polys.append(geom)
    else:
        # 도: 시/군 단위로 병합 (구로 나뉜 시는 시 이름으로 병합)
        city = name[:name.index("시") + 1] if "시" in name else name
        city_groups.setdefault(city, []).append(geom)

# 도 시/군 병합
for city, geoms in city_groups.items():
    merged = unary_union(geoms)
    polys.append(merged)
    names[city] = merged

# 시도 이름 (전체 병합 후 가운데 위치)
sido_names = {}   # 시도 코드 -> (이름, 병합 폴리곤)
for code, geoms in sido_groups.items():
    if code in SIDO_NAMES:
        sido_names[code] = (SIDO_NAMES[code], unary_union(geoms))

# 시도 경계 (unary_union으로 합쳐 외곽선 추출)
sido_boundaries = []
for code, geoms in sido_groups.items():
    merged = unary_union(geoms)
    sido_boundaries.append(merged)

print(f"남한 시군구 {len(polys)}개, 시도 {len(sido_names)}개")

In [ ]:
# 4. 전체 범위 계산 (남한 + 북한 절단선 아래)
xs, ys = [], []
for g in polys:
    minx, miny, maxx, maxy = g.bounds
    xs += [minx, maxx]
    ys += [miny, maxy]

# 북한 범위 포함 (절단선 아래만)
with open(NK, encoding="utf-8") as f:
    nk_gj = json.load(f)
nk_geom = shape(nk_gj["geometry"])
nk_clip = nk_geom.intersection(box(-180, -90, 180, NK_CUT))
nk_b = nk_clip.bounds
xs += [nk_b[0], nk_b[2]]
ys += [nk_b[1], nk_b[3]]
xmin, xmax = min(xs), max(xs)
ymin, ymax = min(ys), max(ys)

# 포항 동쪽 여백 추가 (오른쪽 끝에 공간)
xpad = (xmax - xmin) * 0.03
xmax += xpad

print(f"범위: lon {xmin:.2f}~{xmax:.2f}, lat {ymin:.2f}~{ymax:.2f}")

In [ ]:
# 5. 지도 그리기
fig, ax = plt.subplots(figsize=(W_INCH, H_INCH), dpi=DPI)
# 실제 지리 비율(1.391)로 왜곡 없이 그리기 위해 aspect 조정
# aspect = 실제비율 / 좌표비율 = 1.391 / 1.127 = 1.234
ax.set_aspect(1.391 / 1.127)

# 배경 (바다)
ax.set_facecolor("#cfe8f7")
fig.set_facecolor("#cfe8f7")

# 북한 하단 형태 (연한색, 경계선 없음) - 절단선 아래만
g = nk_clip
if g.geom_type == "Polygon":
    polys_list = [g]
else:
    polys_list = list(g.geoms)
for poly in polys_list:
    x, y = poly.exterior.xy
    ax.fill(x, y, facecolor="#e8f4e8", edgecolor="none", zorder=1)

# 시군구 경계 채우기 (연한색) + 경계선
for g in polys:
    if g.geom_type == "Polygon":
        polys_list = [g]
    else:
        polys_list = list(g.geoms)
    for poly in polys_list:
        x, y = poly.exterior.xy
        ax.fill(x, y, facecolor="#e8f4e8", edgecolor="#333333",
                linewidth=0.5, zorder=2)
        ax.plot(x, y, color="#333333", linewidth=0.5, zorder=3)

# 시도 경계 두껍게
for g in sido_boundaries:
    if g.geom_type == "Polygon":
        polys_list = [g]
    else:
        polys_list = list(g.geoms)
    for poly in polys_list:
        x, y = poly.exterior.xy
        ax.plot(x, y, color="#111111", linewidth=3.0, zorder=4)

# 시/군 이름 표기
for name, g in names.items():
    pt = g.representative_point()
    ax.text(pt.x, pt.y, name, fontsize=24, ha="center", va="center",
            color="#222222", zorder=5)

# 시도 이름 표기 (특별시/광역시: 병하게, 도: 흐리게 2배)
for code, (sname, g) in sido_names.items():
    pt = g.centroid
    if code in METRO_CODES:
        ax.text(pt.x, pt.y, sname, fontsize=24, ha="center", va="center",
                color="#222222", zorder=5)
    else:
        ax.text(pt.x, pt.y, sname, fontsize=48, ha="center", va="center",
                color="#bbbbbb", zorder=4)

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.axis("off")

plt.tight_layout(pad=0)
plt.savefig(OUT, dpi=DPI, facecolor=fig.get_facecolor())
print("saved:", OUT)

## 결과 확인

생성된 `south_korea_map.png` 파일을 확인하세요.

- 크기: 39370 x 54763 px (1000dpi, 1m x 1.391m)
- 파일 크기: 약 39MB

## 설정 변경

2번 셀에서 다음 값을 변경할 수 있습니다:
- `DPI` : 해상도 (기본 1000)
- `W_INCH` : 가로 크기 (기본 39.37인치 = 1m)
- `NK_CUT` : 북한 절단 위도
- `SIDO_NAMES` : 시도 이름
- 색상: `#cfe8f7`(바다), `#e8f4e8`(육지), `#333333`(경계선), `#111111`(시도 경계)